# Nemotron-3-Diarization benchmark on Colab
**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom.
Untested in Colab; if an install cell fails, use the exact install line from the model card (https://huggingface.co/nvidia/Nemotron-3-Diarization).

## 1. Check the GPU

In [1]:
!nvidia-smi | head -15
import torch; print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch the runtime to a GPU (T4)'

Tue Sep 29 16:16:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload `nemotron_bench.zip`

In [2]:
from google.colab import files
import os, zipfile
up = files.upload()  # pick nemotron_bench.zip
zipfile.ZipFile(next(iter(up))).extractall('/content')
%cd /content/nemotron_bench
!ls -A

Saving nemotron_bench.zip to nemotron_bench (1).zip
/content/nemotron_bench
app.py	core.py  diagnostics.py  README.md  requirements.txt  .streamlit


In [3]:
# Install NeMo from GitHub main, on top of the deps you already have
!pip -q install --no-deps --force-reinstall "git+https://github.com/NVIDIA-NeMo/Speech.git@main"
!pip -q install "setuptools<82" "protobuf>=6.31.1,<7"
!pip show nemo-toolkit | head -3

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Name: nemo-toolkit
Version: 3.1.0+9bed1bb96
Summary: NeMo - a toolkit for Conversational AI
ERROR: Pipe to stdout was broken


## 3. Install (several minutes). If it asks to restart the runtime, do it, then re-run cell 2's `%cd` line.

In [4]:
!apt-get -qq install -y ffmpeg libsndfile1
!pip -q install -U pip wheel Cython packaging "setuptools<82"
!pip -q install -r requirements.txt
!pip -q install --no-deps --force-reinstall "git+https://github.com/NVIDIA-NeMo/Speech.git@main"
!pip -q install "setuptools<82" "protobuf>=6.31.1,<7"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lightning 2.4.0 requires packaging<25.0,>=20.0, but you have packaging 26.3 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 4. Hugging Face login (only needed if the model download is refused)

In [5]:
from huggingface_hub import login
login()

## 5. Smoke check: NeMo imports and the model loads on GPU

In [6]:
import sys; sys.path.insert(0, '/content/nemotron_bench')
from nemo.collections.asr.models import SortformerEncLabelModel
m = SortformerEncLabelModel.from_pretrained('nvidia/Nemotron-3-Diarization').to('cuda').eval()
print('model loaded OK'); del m
import torch; torch.cuda.empty_cache()

Nemotron-3-Diarization.nemo: reconstructing file:   0%|          |  0.00B /  199MB            

Nemotron-3-Diarization.nemo: downloading bytes:           |  0.00B            

[NeMo W 2026-09-29 16:19:37 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    validate_manifest_paths: false
    sample_rate: 16000
    num_spks: 8
    session_len_sec: 105
    soft_targets: false
    batch_size: 42
    shuffle: true
    num_workers: 18
    pin_memory: true
    use_lhotse: false
    subsegment_mode: true
    subsegment_single_chunk_min_len_sec: 45
    subsegment_two_chunks_rate: 0.5
    subsegment_two_chunk_min_len_sec: 22
    subsegment_nspk_bias: 1.5
    subsegment_start_guard_sec: 0.25
    subsegment_min_first_spk_sec: 0.5
    subsegment_splice_silence_sec: 0.1
    
[NeMo W 2026-09-29 16:19:37 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the

[NeMo I 2026-09-29 16:19:39 save_restore_connector:288] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--Nemotron-3-Diarization/snapshots/f667ed73aee57d40cc39428eb768b4fd87a0a29e/Nemotron-3-Diarization.nemo.
model loaded OK


## 6. Start the app and a public tunnel

In [7]:
import subprocess, time, re, os
os.chdir('/content/nemotron_bench')
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared', shell=True)
subprocess.Popen('nohup streamlit run app.py --server.port 8501 --server.headless true > st.log 2>&1 &', shell=True)
time.sleep(8)
subprocess.Popen('nohup ./cloudflared tunnel --url http://localhost:8501 > cf.log 2>&1 &', shell=True)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('cf.log').read())
    if m: url = m.group(0); break
print('OPEN THIS:', url or 'no URL yet, see cf.log')

OPEN THIS: https://tablets-fascinating-mardi-passed.trycloudflare.com


Open the link above. **Anyone with the link can use the app**, so don't share it. If the page shows an error, check the logs below.

In [8]:
!tail -20 /content/nemotron_bench/st.log



2026-09-29 16:19:42.960 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.110.30.137:8501



## 7. Save results to Google Drive (Colab wipes files when the session ends)
Re-run after each batch of runs.

In [9]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nemotron_runs
!cp -r /content/nemotron_bench/static/runs/. /content/drive/MyDrive/nemotron_runs/
!ls /content/drive/MyDrive/nemotron_runs

Mounted at /content/drive
cp: cannot stat '/content/nemotron_bench/static/runs/.': No such file or directory


## 8. Optional: download everything as a zip

In [10]:
!cd /content/nemotron_bench/static && zip -qr /content/runs.zip runs
from google.colab import files; files.download('/content/runs.zip')

/bin/bash: line 1: cd: /content/nemotron_bench/static: No such file or directory


FileNotFoundError: Cannot find file: /content/runs.zip

## Notes
- Big videos upload slowly through the tunnel. Extract audio first: `ffmpeg -i in.mp4 -vn -ac 1 -ar 16000 out.wav`.
- If the GPU runs out of memory, set the ASR model to `medium` in the app sidebar.
- The tunnel URL changes every time you re-run cell 6.